# LeetCode #1308: Run-Length Encoding II

https://leetcode.com/problems/run-length-encoding-ii/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(n^2)$ | $O(n)$ |
| **Optimal: Two-Pointer Compression ★** | $O(n)$ | $O(n)$ |

---

## Understanding the Methods

### Brute Force
Rebuild the full decoded string from the run-length encoded input, then re-encode after deletion. Decoding can expand to $O(n^2)$ characters and requires $O(n^2)$ space, making it impractical for large counts.

### Optimal: Two-Pointer Compression ★
Process the encoded list directly without decoding. Walk through each `[count, char]` pair, adjusting the count for the character at position `k` (the one to delete). If a run count drops to zero, remove it; if adjacent runs share the same character after adjustment, merge them. One linear pass, $O(n)$ time, $O(n)$ output space.

**Constraints:**
* The encoded list represents a string of length at most $10^5$
* $1 \leq k \leq \text{length of decoded string}$
* Each `[count, char]` pair has count $\geq 1$


## Solutions

### C#

In [ ]:
public class Solution {
    public IList<IList<int>> Encode(IList<IList<int>> encoded, int k) {
        var result = new List<IList<int>>();
        // Scan pairs to locate and remove position k, then rebuild the encoding
        int pos = 0;
        foreach (var pair in encoded) {
            int cnt = pair[0], ch = pair[1];
            if (pos < k && k <= pos + cnt) {
                // Deletion falls inside this run — split it around position k
                if (k - pos - 1 > 0) Append(result, k - pos - 1, ch);
                if (pos + cnt - k > 0) Append(result, pos + cnt - k, ch);
            } else {
                Append(result, cnt, ch);
            }
            pos += cnt;
        }
        return result;
    }

    private void Append(List<IList<int>> res, int cnt, int ch) {
        // Merge into the previous run when the character matches to keep encoding compact
        if (res.Count > 0 && res[^1][1] == ch)
            res[^1][0] += cnt;
        else
            res.Add(new List<int> { cnt, ch });
    }
}

### Python

In [ ]:
class Solution:
    def encode(self, encoded: list[list[int]], k: int) -> list[list[int]]:
        result = []
        pos = 0
        for cnt, ch in encoded:
            if pos < k <= pos + cnt:
                # Deletion falls inside this run — split it around position k
                if k - pos - 1 > 0:
                    self._append(result, k - pos - 1, ch)
                if pos + cnt - k > 0:
                    self._append(result, pos + cnt - k, ch)
            else:
                self._append(result, cnt, ch)
            pos += cnt
        return result

    def _append(self, res: list, cnt: int, ch: int) -> None:
        # Merge into the previous run when the character matches to keep encoding compact
        if res and res[-1][1] == ch:
            res[-1][0] += cnt
        else:
            res.append([cnt, ch])

### Go

In [ ]:
func encode(encoded [][]int, k int) [][]int {
    var result [][]int
    pos := 0
    appendRun := func(cnt, ch int) {
        // Merge into the previous run when the character matches to keep encoding compact
        if len(result) > 0 && result[len(result)-1][1] == ch {
            result[len(result)-1][0] += cnt
        } else {
            result = append(result, []int{cnt, ch})
        }
    }
    for _, pair := range encoded {
        cnt, ch := pair[0], pair[1]
        if pos < k && k <= pos+cnt {
            // Deletion falls inside this run — split it around position k
            if k-pos-1 > 0 { appendRun(k-pos-1, ch) }
            if pos+cnt-k > 0 { appendRun(pos+cnt-k, ch) }
        } else {
            appendRun(cnt, ch)
        }
        pos += cnt
    }
    return result
}

### Rust

In [ ]:
impl Solution {
    pub fn encode(encoded: Vec<Vec<i32>>, k: i32) -> Vec<Vec<i32>> {
        let mut result: Vec<Vec<i32>> = Vec::new();
        let mut pos = 0i32;

        let mut append = |res: &mut Vec<Vec<i32>>, cnt: i32, ch: i32| {
            // Merge into the previous run when the character matches to keep encoding compact
            if let Some(last) = res.last_mut() {
                if last[1] == ch { last[0] += cnt; return; }
            }
            res.push(vec![cnt, ch]);
        };

        for pair in &encoded {
            let (cnt, ch) = (pair[0], pair[1]);
            if pos < k && k <= pos + cnt {
                // Deletion falls inside this run — split it around position k
                if k - pos - 1 > 0 { append(&mut result, k - pos - 1, ch); }
                if pos + cnt - k > 0 { append(&mut result, pos + cnt - k, ch); }
            } else {
                append(&mut result, cnt, ch);
            }
            pos += cnt;
        }
        result
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `encoded = [[1,1],[2,2],[3,1],[1,2]]`, `k = 2`
Decoded string: `[1,2,2,1,1,1,2]`. Deleting position 2 (value 2) from a run of 2 splits the `[2,2]` into `[1,2]`. After split, adjacent `[1,1]` and `[1,1]` merge into `[2,1]`. Result: `[[1,1],[1,2],[2,1],[1,2]]`.

### 2. Slightly Complex
**Input:** `encoded = [[3,1],[3,2]]`, `k = 3`
Deletion at position 3 removes the last character of the first run. Count drops from 3 to 2: `[[2,1],[3,2]]`. No merging needed — characters differ across the boundary.

### 3. Edge Case: Time Factor
**Input:** encoded list has 50,000 pairs; `k` falls in the very last pair
Every pair is scanned before the deletion is found. $O(n)$ traversal is unavoidable, confirming the linear lower bound.

### 4. Edge Case: Space Factor
**Input:** `encoded = [[100000,1]]`, `k = 50000`
Single run split into two equal runs: `[[49999,1],[50000,1]]`. No merging possible. Output size doubles from 1 pair to 2 — maximum output growth from a single deletion.

### 5. Almost-Impossible but Plausible
**Input:** `encoded = [[1,1],[1,2],[1,1]]`, `k = 2`
Deleting the lone `2` leaves `[[1,1],[1,1]]`. The merge step fuses both runs into `[[2,1]]`. Without the merge check, the output would be invalid (consecutive runs of the same character).
